In [1]:
from huggingface_hub import login
import pandas as pd
from datasets import load_dataset
import zstandard as zstd
import requests
import io
import json

login()

/home/g_husky/code/DATA_PRIVACY/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load the data from Hugginface. We use two different repos: one for the train split, one for the test split

In [4]:

TRAIN_ROWS_NEEDED = 5000 # True Members
TEST_ROWS_NEEDED = 5000   # True Non-Members
DOMAIN = "PubMed Abstracts" # domain

# TRAIN DATA (MEMBERS)
# Method: Hugging Face Datasets via Parquet
# https://huggingface.co/datasets/monology/pile-uncopyrighted


print(f"--- Fetching {TRAIN_ROWS_NEEDED} Train Rows (Members) ---")

# 1. Load the streaming Parquet branch
train_dataset = load_dataset(
    "monology/pile-uncopyrighted", 
    split="train",
    revision="refs/convert/parquet",
    streaming=True
)

# 2. Filter and extract
def is_domain(example):
    return example["meta"]["pile_set_name"] == DOMAIN

domain_train_stream = train_dataset.filter(is_domain)
train_members_list = list(domain_train_stream.take(TRAIN_ROWS_NEEDED))

# 3. Save to Parquet
df_train = pd.DataFrame(train_members_list)
df_train.to_parquet("../dataset/members.parquet")
print(f"Saved {len(df_train)} rows to dataset/members.parquet\n")


# TEST DATA (NON-MEMBERS)
# Method: Direct zstandard streaming
# https://huggingface.co/datasets/monology/pile-test-val

print(f"--- Fetching {TEST_ROWS_NEEDED} Test Rows (Non-Members) ---")

def stream_test_set(max_rows, domain):
    url = "https://huggingface.co/datasets/mit-han-lab/pile-val-backup/resolve/main/val.jsonl.zst"
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    dctx = zstd.ZstdDecompressor()
    stream_reader = dctx.stream_reader(response.raw)
    text_stream = io.TextIOWrapper(stream_reader, encoding='utf-8')
    
    extracted_data = []
    
    for line in text_stream:
        row = json.loads(line)
        if row.get("meta", {}).get("pile_set_name") == domain:
            extracted_data.append(row)
            if len(extracted_data) >= max_rows:
                break
                
    return extracted_data

# 1. Stream and extract
test_non_members_list = stream_test_set(TEST_ROWS_NEEDED, DOMAIN)

# 2. Save to Parquet
df_test = pd.DataFrame(test_non_members_list)
df_test.to_parquet("../dataset/non_members.parquet")
print(f"Saved {len(df_test)} rows to dataset/non_members.parquet\n")


# VERIFICATION
print("--- Final Verification ---")
check_train = pd.read_parquet("../dataset/members.parquet")
check_test = pd.read_parquet("../dataset/non_members.parquet")

print(f"Members Dataset (Train): {check_train.shape[0]} rows")
print(f"Non-Members Dataset (Test): {check_test.shape[0]} rows")

--- Fetching 5000 Train Rows (Members) ---
Saved 5000 rows to dataset/members.parquet

--- Fetching 5000 Test Rows (Non-Members) ---
Saved 5000 rows to dataset/non_members.parquet

--- Final Verification ---
Members Dataset (Train): 5000 rows
Non-Members Dataset (Test): 5000 rows


In [5]:
members = pd.read_parquet("../dataset/members.parquet")
non_members = pd.read_parquet("../dataset/non_members.parquet")

non_members.head()

,text,meta
0,"Effect of sleep quality on memory, executive f...",{'pile_set_name': 'PubMed Abstracts'}
1,Fluorescent labeling of both GABAergic and gly...,{'pile_set_name': 'PubMed Abstracts'}
2,"Carotid endarterectomy: operative risks, recur...",{'pile_set_name': 'PubMed Abstracts'}
3,Regulation of the anaerobic metabolism in Baci...,{'pile_set_name': 'PubMed Abstracts'}
4,Early and long-term outcomes after manual and ...,{'pile_set_name': 'PubMed Abstracts'}


## 1. check if there are ovelaps between members and non-members (data leakage).

In [6]:
def check_sample_overlap(members_text, non_members_text):
    """
    Checks for identical text samples shared between members and non-members.
    """
    # Normalize by stripping leading/trailing whitespace to catch hidden duplicates
    m_set = set(str(t).strip() for t in members_text)
    nm_set = set(str(t).strip() for t in non_members_text)
    
    # Find the intersection (shared samples)
    overlap = m_set.intersection(nm_set)
    num_overlap = len(overlap)
    
    print("=" * 50)
    print(" DATA LEAKAGE CHECK")
    print("=" * 50)
    print(f"• Members:              {len(m_set):,} unique samples")
    print(f"• Non-Members:          {len(nm_set):,} unique samples")
    print(f"• Overlapping Samples:  {num_overlap:,}")
    
    if num_overlap > 0:
        print("\n=> WARNING: Contamination detected!")
        print(f"   {num_overlap} samples exist in both training and test sets.")
        print("   These must be removed from the non-members set to maintain a valid MIA.")
    else:
        print("\n=> SUCCESS: No overlap detected. Splits are strictly separated.")


    return overlap


# Run the check
shared_samples = check_sample_overlap(members["text"], non_members["text"])

# If you want to see what the first overlapping text actually looks like:
if shared_samples:
    print("\nSample overlap text:")
    print(list(shared_samples)[0][:200] + "...")

 DATA LEAKAGE CHECK
• Members:              5,000 unique samples
• Non-Members:          5,000 unique samples
• Overlapping Samples:  0

=> SUCCESS: No overlap detected. Splits are strictly separated.


## 2. compare the length of members and non-members

In [7]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ==========================================
# HELPER FUNCTIONS
# ==========================================

def _default_tokenizer(text):
    """Fast fallback word-level tokenizer."""
    return re.findall(r"\b\w+\b", str(text).lower())

m_tokens = _default_tokenizer(members)
nm_tokens = _default_tokenizer(non_members)

# ==========================================
# 1. LENGTH ANALYSIS
# ==========================================

def analyze_length_distribution(m_tokens, nm_tokens):
    print("\n[1] LENGTH & TOKEN COUNT DISTRIBUTION")
    print("-" * 65)

    m_lengths = np.array([len(t) for t in m_tokens])
    nm_lengths = np.array([len(t) for t in nm_tokens])

    len_summary = pd.DataFrame({
        "Metric": ["Count", "Mean", "Std", "Median", "IQR", "Min", "Max"],
        "Members (Train)": [
            len(m_lengths), np.mean(m_lengths), np.std(m_lengths),
            np.median(m_lengths), stats.iqr(m_lengths), np.min(m_lengths), np.max(m_lengths)
        ],
        "Non-Members (Test)": [
            len(nm_lengths), np.mean(nm_lengths), np.std(nm_lengths),
            np.median(nm_lengths), stats.iqr(nm_lengths), np.min(nm_lengths), np.max(nm_lengths)
        ]
    })
    print(len_summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

    ks_stat, ks_pval = stats.ks_2samp(m_lengths, nm_lengths)
    mwu_stat, mwu_pval = stats.mannwhitneyu(m_lengths, nm_lengths, alternative="two-sided")

    print("\nStatistical Tests on Length:")
    print(f"  • Kolmogorov-Smirnov Test: stat = {ks_stat:.4f}, p-value = {ks_pval:.4e}")
    print(f"  • Mann-Whitney U Test:     stat = {mwu_stat:.1f}, p-value = {mwu_pval:.4e}")
    
    if ks_pval > 0.05:
        print("  => Verdict: Fail to reject H0. Length distributions are statistically indistinguishable (p > 0.05).")
    else:
        print("  => Warning: Length distributions show statistically significant variance (p <= 0.05).")

    return m_lengths, nm_lengths, len_summary, ks_pval, mwu_pval


analyze_length_distribution(m_tokens, nm_tokens)




[1] LENGTH & TOKEN COUNT DISTRIBUTION
-----------------------------------------------------------------
Metric  Members (Train)  Non-Members (Test)
 Count           119.00              127.00
  Mean             6.38                5.89
   Std             3.82                3.67
Median             6.00                5.00
   IQR             5.50                6.00
   Min             1.00                1.00
   Max            13.00               14.00

Statistical Tests on Length:
  • Kolmogorov-Smirnov Test: stat = 0.0852, p-value = 7.2015e-01
  • Mann-Whitney U Test:     stat = 8091.5, p-value = 3.3505e-01
  => Verdict: Fail to reject H0. Length distributions are statistically indistinguishable (p > 0.05).


(array([ 4,  1,  3, 11,  5,  9,  8,  4,  1,  4,  9,  3,  4, 12,  8,  1, 12,
         8,  3,  7,  6,  5,  1, 11,  2,  7,  9,  4,  8,  1, 11,  3, 11,  2,
         1, 13,  4, 10,  2, 11, 10,  5,  2,  4, 10,  2,  7,  4,  6,  7,  3,
         4,  4,  9,  8, 10,  2,  2,  1,  3,  4, 12, 11,  2, 11,  6,  4, 11,
         2,  6, 12,  2,  8,  4,  1, 13,  6,  9,  1, 13,  6,  9,  1, 13,  6,
         9,  1, 13,  6,  9,  1, 13,  6,  9,  4, 13,  6,  9,  4, 13,  6,  9,
         4, 13,  6,  9,  4, 13,  6,  9,  4, 13,  6,  9,  4,  4,  1,  1,  7]),
 array([ 4,  1,  6,  2,  5,  7,  2,  6,  9,  1,  1, 11,  8,  2,  4,  9,  3,
         3,  1,  7, 14,  9,  5,  5,  1, 10,  2,  3,  9, 10,  2,  4,  1,  5,
         3,  4,  4,  8,  5,  6,  3,  4,  5,  7,  9,  5,  7,  2,  5,  4,  8,
         9,  2,  4,  2,  1,  3,  3,  6,  4,  8, 10,  2,  3,  8,  7,  1,  4,
        13,  3,  8, 13,  2,  2,  4,  7,  2, 14,  8,  4,  4,  4,  1, 13,  6,
         9,  1, 13,  6,  9,  1, 13,  6,  9,  1, 13,  6,  9,  1, 13,  6,  9,
         4

In [ ]:
# # ==========================================
# # 2. TEMPORAL ANALYSIS
# # ==========================================

# def analyze_temporal_distribution(m_text, nm_text):
#     print("\n[2] TEMPORAL DISTRIBUTION (Publication Years)")
#     print("-" * 65)

#     year_pattern = re.compile(r"\b(19\d\d|20[0-2]\d)\b")

#     def extract_years(texts):
#         years = []
#         for doc in texts:
#             found = [int(y) for y in year_pattern.findall(str(doc))]
#             years.extend(found)
#         return np.array(years)

#     m_years = extract_years(m_text)
#     nm_years = extract_years(nm_text)

#     if len(m_years) > 0 and len(nm_years) > 0:
#         year_summary = pd.DataFrame({
#             "Metric": ["Mentions Extracted", "Earliest", "25th Pct", "Median", "75th Pct", "Latest"],
#             "Members": [
#                 len(m_years), int(np.min(m_years)), int(np.percentile(m_years, 25)),
#                 int(np.median(m_years)), int(np.percentile(m_years, 75)), int(np.max(m_years))
#             ],
#             "Non-Members": [
#                 len(nm_years), int(np.min(nm_years)), int(np.percentile(nm_years, 25)),
#                 int(np.median(nm_years)), int(np.percentile(nm_years, 75)), int(np.max(nm_years))
#             ]
#         })
#         print(year_summary.to_string(index=False))
#         ks_year_stat, ks_year_pval = stats.ks_2samp(m_years, nm_years)
#         print(f"\nTemporal KS Test: stat = {ks_year_stat:.4f}, p-value = {ks_year_pval:.4e}")
#     else:
#         print("  No 4-digit years found in abstract texts.")

#     return m_years, nm_years


# # analyze_temporal_distribution(members, non_members)


In [8]:
# ==========================================
# 3. SURFACE STATISTICS
# ==========================================

def analyze_surface_statistics(m_tokens, nm_tokens, m_text, nm_text):
    print("\n[4] SURFACE STATISTICS & STRUCTURAL MARKERS")
    print("-" * 65)

    m_flat_tokens = [t for doc in m_tokens for t in doc]
    nm_flat_tokens = [t for doc in nm_tokens for t in doc]

    m_vocab, nm_vocab = set(m_flat_tokens), set(nm_flat_tokens)

    m_ttr = len(m_vocab) / len(m_flat_tokens) if m_flat_tokens else 0
    nm_ttr = len(nm_vocab) / len(nm_flat_tokens) if nm_flat_tokens else 0

    intersection = m_vocab.intersection(nm_vocab)
    union = m_vocab.union(nm_vocab)
    jaccard = len(intersection) / len(union) if union else 0
    overlap_coeff = len(intersection) / min(len(m_vocab), len(nm_vocab)) if min(len(m_vocab), len(nm_vocab)) else 0

    print("Vocabulary Metrics:")
    print(f"  • Members Unique Types:     {len(m_vocab):,}")
    print(f"  • Non-Members Unique Types: {len(nm_vocab):,}")
    print(f"  • Type-Token Ratio (M):     {m_ttr:.4f}")
    print(f"  • Type-Token Ratio (NM):    {nm_ttr:.4f}")
    print(f"  • Jaccard Similarity:       {jaccard:.4f}")
    print(f"  • Overlap Coefficient:      {overlap_coeff:.4f}")

    markers = {
        "HTML/XML Tags": re.compile(r"<[^>]+>"),
        "Markdown Headers": re.compile(r"^#{1,6}\s", re.MULTILINE),
        "Code Syntax/Braces": re.compile(r"[{};]\s*$|def\s|class\s", re.MULTILINE),
        "Abstract Header": re.compile(r"\babstract\b", re.IGNORECASE),
        "Methods Header": re.compile(r"\bmethods\b", re.IGNORECASE),
        "Results Header": re.compile(r"\bresults\b", re.IGNORECASE),
        "Conclusions Header": re.compile(r"\bconclusions?\b", re.IGNORECASE),
        "DOI / PMID Identifiers": re.compile(r"\b(doi|pmid)\b", re.IGNORECASE),
    }

    marker_rates = []
    for name, pattern in markers.items():
        m_matches = sum(1 for doc in m_text if pattern.search(str(doc))) / len(m_text)
        nm_matches = sum(1 for doc in nm_text if pattern.search(str(doc))) / len(nm_text)
        marker_rates.append({
            "Structural Marker": name,
            "Members Rate": m_matches,
            "Non-Members Rate": nm_matches,
            "Abs Diff": abs(m_matches - nm_matches)
        })

    df_markers = pd.DataFrame(marker_rates)
    print("\nMarker Prevalence (% of documents containing marker):")
    print(df_markers.to_string(index=False, formatters={
        "Members Rate": "{:.2%}".format,
        "Non-Members Rate": "{:.2%}".format,
        "Abs Diff": "{:.2%}".format
    }))

    return df_markers, m_ttr, nm_ttr, jaccard


analyze_surface_statistics(m_tokens, nm_tokens, members, non_members)


[4] SURFACE STATISTICS & STRUCTURAL MARKERS
-----------------------------------------------------------------
Vocabulary Metrics:
  • Members Unique Types:     35
  • Non-Members Unique Types: 35
  • Type-Token Ratio (M):     0.0461
  • Type-Token Ratio (NM):    0.0468
  • Jaccard Similarity:       0.9444
  • Overlap Coefficient:      0.9714

Marker Prevalence (% of documents containing marker):
     Structural Marker Members Rate Non-Members Rate Abs Diff
         HTML/XML Tags        0.00%            0.00%    0.00%
      Markdown Headers        0.00%            0.00%    0.00%
    Code Syntax/Braces        0.00%            0.00%    0.00%
       Abstract Header        0.00%            0.00%    0.00%
        Methods Header        0.00%            0.00%    0.00%
        Results Header        0.00%            0.00%    0.00%
    Conclusions Header        0.00%            0.00%    0.00%
DOI / PMID Identifiers        0.00%            0.00%    0.00%


(        Structural Marker  Members Rate  Non-Members Rate  Abs Diff
 0           HTML/XML Tags           0.0               0.0       0.0
 1        Markdown Headers           0.0               0.0       0.0
 2      Code Syntax/Braces           0.0               0.0       0.0
 3         Abstract Header           0.0               0.0       0.0
 4          Methods Header           0.0               0.0       0.0
 5          Results Header           0.0               0.0       0.0
 6      Conclusions Header           0.0               0.0       0.0
 7  DOI / PMID Identifiers           0.0               0.0       0.0,
 0.0461133069828722,
 0.04679144385026738,
 0.9444444444444444)

In [25]:
# ==========================================
# 4. VISUALIZATIONS
# ==========================================

def plot_diagnostic_visualizations(m_lengths, nm_lengths, m_years, nm_years, df_markers, figsize=(16, 12)):
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=figsize)

    # 1. Length KDE
    sns.kdeplot(m_lengths, ax=axes[0, 0], label="Members", fill=True, color="#1f77b4", alpha=0.3)
    sns.kdeplot(nm_lengths, ax=axes[0, 0], label="Non-Members", fill=True, color="#ff7f0e", alpha=0.3)
    axes[0, 0].set_title("Token Count Distribution (Density)", fontsize=13, fontweight="bold")
    axes[0, 0].set_xlabel("Tokens per Document")
    axes[0, 0].legend()

    # 2. Length QQ
    p_steps = np.linspace(1, 99, 99)
    m_quants = np.percentile(m_lengths, p_steps)
    nm_quants = np.percentile(nm_lengths, p_steps)
    axes[0, 1].plot(m_quants, nm_quants, "o", color="#2ca02c", alpha=0.7)
    max_val = max(np.max(m_quants), np.max(nm_quants))
    axes[0, 1].plot([0, max_val], [0, max_val], "--r", label="Ideal Identity (1:1)")
    axes[0, 1].set_title("Q-Q Length Quantiles Comparison", fontsize=13, fontweight="bold")
    axes[0, 1].set_xlabel("Members Percentiles (1st-99th)")
    axes[0, 1].set_ylabel("Non-Members Percentiles (1st-99th)")
    axes[0, 1].legend()

    # 3. Temporal Distribution
    if len(m_years) > 0 and len(nm_years) > 0:
        year_bins = np.arange(1970, 2025, 2)
        sns.histplot(m_years, ax=axes[1, 0], bins=year_bins, stat="density", element="step",
                     color="#1f77b4", label="Members", alpha=0.5)
        sns.histplot(nm_years, ax=axes[1, 0], bins=year_bins, stat="density", element="step",
                     color="#ff7f0e", label="Non-Members", alpha=0.5)
        axes[1, 0].set_title("Extracted Year Mentions (1970 - 2024)", fontsize=13, fontweight="bold")
        axes[1, 0].set_xlabel("Publication Year")
        axes[1, 0].set_xlim(1975, 2024)
        axes[1, 0].legend()

    # 4. Marker Barplot
    marker_plot_df = pd.melt(
        df_markers, id_vars=["Structural Marker"], 
        value_vars=["Members Rate", "Non-Members Rate"],
        var_name="Split", value_name="Rate"
    )
    sns.barplot(data=marker_plot_df, y="Structural Marker", x="Rate", hue="Split", ax=axes[1, 1], palette="Set2")
    axes[1, 1].set_title("Structural Marker Prevalence", fontsize=13, fontweight="bold")
    axes[1, 1].set_xlabel("Proportion of Documents")
    axes[1, 1].set_ylabel("")

    plt.tight_layout()
    plt.show()

In [ ]:
# # ==========================================
# # 3. DOMAIN VERIFICATION
# # ==========================================

# def verify_domain_source(m_meta, nm_meta):
#     print("\n[3] DOMAIN & SOURCE VERIFICATION")
#     print("-" * 65)

#     def parse_pile_set_name(meta_val):
#         if isinstance(meta_val, dict):
#             return meta_val.get("pile_set_name", "Unknown")
#         elif isinstance(meta_val, str):
#             try:
#                 import ast
#                 return ast.literal_eval(meta_val).get("pile_set_name", "Unknown")
#             except Exception:
#                 return "Unknown"
#         return "Not Provided"

#     if m_meta is not None:
#         m_domains = m_meta.apply(parse_pile_set_name).value_counts(normalize=True).to_dict()
#         print(f"Members Metadata Origin:     {m_domains}")
#     else:
#         print("Members Metadata Origin:     Provenance-confirmed via upstream filter.")

#     if nm_meta is not None:
#         nm_domains = nm_meta.apply(parse_pile_set_name).value_counts(normalize=True).to_dict()
#         print(f"Non-Members Metadata Origin: {nm_domains}")
#     else:
#         print("Non-Members Metadata Origin: Provenance-confirmed via upstream filter.")

#     print("\nConfirmation Method:")
#     print("  • Splits were extracted using deterministic metadata queries matching 'PubMed Abstracts'.")




# ==========================================
# MAIN ORCHESTRATOR
# ==========================================

def audit_dataset_distribution(members_data, non_members_data, tokenizer_func=None, figsize=(16, 12)):
    """Main orchestrator function that calls all sub-audits."""
    print("=" * 65)
    print("      MEMBER vs. NON-MEMBER DISTRIBUTION AUDIT REPORT")
    print("=" * 65)

    m_text, m_meta = _extract_text_and_meta(members_data)
    nm_text, nm_meta = _extract_text_and_meta(non_members_data)

    tokenizer = tokenizer_func or _default_tokenizer
    m_tokens = [tokenizer(doc) for doc in m_text]
    nm_tokens = [tokenizer(doc) for doc in nm_text]

    # Run the modular parts
    m_len, nm_len, len_summary, ks_pval, mwu_pval = analyze_length_distribution(m_tokens, nm_tokens)
    m_yrs, nm_yrs = analyze_temporal_distribution(m_text, nm_text)
    verify_domain_source(m_meta, nm_meta)
    df_markers, m_ttr, nm_ttr, jaccard = analyze_surface_statistics(m_tokens, nm_tokens, m_text, nm_text)
    
    # Render Plots
    plot_diagnostic_visualizations(m_len, nm_len, m_yrs, nm_yrs, df_markers, figsize=figsize)

    return {
        "length_summary": len_summary,
        "ks_length_pval": ks_pval,
        "mwu_length_pval": mwu_pval,
        "ttr_members": m_ttr,
        "ttr_non_members": nm_ttr,
        "jaccard_vocab": jaccard,
        "marker_comparison": df_markers
    }

In [33]:
def check_sample_overlap(members_text, non_members_text):
    """
    Checks for identical text samples shared between members and non-members.
    """
    # Normalize by stripping leading/trailing whitespace to catch hidden duplicates
    m_set = set(str(t).strip() for t in members_text)
    nm_set = set(str(t).strip() for t in non_members_text)
    
    # Find the intersection (shared samples)
    overlap = m_set.intersection(nm_set)
    num_overlap = len(overlap)
    
    print("=" * 50)
    print(" DATA LEAKAGE CHECK")
    print("=" * 50)
    print(f"• Members:              {len(m_set):,} unique samples")
    print(f"• Non-Members:          {len(nm_set):,} unique samples")
    print(f"• Overlapping Samples:  {num_overlap:,}")
    
    if num_overlap > 0:
        print("\n=> WARNING: Contamination detected!")
        print(f"   {num_overlap} samples exist in both training and test sets.")
        print("   These must be removed from the non-members set to maintain a valid MIA.")
    else:
        print("\n=> SUCCESS: No overlap detected. Splits are strictly separated.")
        
    return overlap

check_sample_overlap(members, non_members)

 DATA LEAKAGE CHECK
• Members:              2 unique samples
• Non-Members:          2 unique samples
• Overlapping Samples:  2

=> WARNING: Contamination detected!
   2 samples exist in both training and test sets.
   These must be removed from the non-members set to maintain a valid MIA.


{'meta', 'text'}